# 실습 06 · SPC 관리도 + 다변량 이상탐지
### 전통적 품질관리(SPC)와 AI 이상탐지를 함께

**왜 중요한가 (현장 맥락)**
GMP 제조에서 **통계적 공정관리(SPC, 관리도)** 는 필수입니다.
하지만 관리도는 변수 하나씩만 봅니다. 실제 공정은 **여러 변수가 얽혀** 있어,
각 변수는 정상 범위인데 **조합이 비정상**인 경우를 놓칩니다.
→ 여기서 **AI 다변량 이상탐지**가 SPC 를 보완합니다.


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
np.random.seed(0)

# 정제 타정(tableting) 공정을 모사: 정상 300배치 + 이상 몇 개
n = 300
weight   = np.random.normal(500, 4, n)    # 정제 중량(mg)
hardness = np.random.normal(80, 5, n)     # 경도(N)
# 중량과 경도는 서로 양의 상관 (압축력 때문)
hardness += 0.5*(weight-500)
thick    = np.random.normal(4.0, 0.05, n) # 두께(mm)

df = pd.DataFrame({"중량":weight,"경도":hardness,"두께":thick})
# 이상치 5개 주입 (개별값은 정상범위지만 '조합'이 깨진 경우 포함)
idx = [50, 120, 200, 250, 280]
df.loc[50,"경도"]  = 55     # 중량 정상인데 경도만 급락
df.loc[120,"중량"] = 514    # 중량 상한
df.loc[200,["중량","경도"]] = [498, 100]  # 상관관계 위반
df.loc[250,"두께"] = 4.25
df.loc[280,["중량","경도"]] = [502, 60]
df.head()

## 1. 전통적 SPC — X-관리도 (I-chart)
평균±3σ 를 관리한계(UCL/LCL)로 정하고, 벗어나면 경보합니다.


In [ ]:
def control_chart(series, name):
    mu, sd = series.mean(), series.std()
    ucl, lcl = mu+3*sd, mu-3*sd
    plt.figure(figsize=(9,3))
    plt.plot(series.values, marker="o", ms=3, color="#4C72B0")
    plt.axhline(mu, color="green", label="중심선(CL)")
    plt.axhline(ucl, color="red", ls="--", label="±3σ 관리한계")
    plt.axhline(lcl, color="red", ls="--")
    out = series[(series>ucl)|(series<lcl)]
    plt.scatter(out.index, out.values, color="red", zorder=5, s=60)
    plt.title(f"{name} 관리도  (이탈 {len(out)}건)"); plt.legend(loc="upper right")
    plt.tight_layout(); plt.show()
    return set(out.index)

flagged = set()
for c in ["중량","경도","두께"]:
    flagged |= control_chart(df[c], c)
print("SPC 가 잡아낸 이상 배치 index:", sorted(flagged))

**관찰**: SPC 는 개별 변수가 한계를 벗어난 경우만 잡습니다.
index 200, 280 처럼 **각 값은 정상이지만 중량-경도 상관이 깨진** 경우는 놓치기 쉽습니다.
→ 다변량 방법이 필요합니다.


## 2. ⭐ 다변량 이상탐지 (Isolation Forest)
여러 변수를 **동시에** 보고 "정상 패턴에서 얼마나 벗어났는지"를 점수화합니다.
`IsolationForest` 는 이상치를 빠르게 격리(isolate)하는 트리 기반 방법입니다.


In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

Xs = StandardScaler().fit_transform(df)
iso = IsolationForest(contamination=0.02, random_state=42).fit(Xs)
df["이상점수"] = -iso.score_samples(Xs)      # 높을수록 이상
df["이상여부"] = (iso.predict(Xs)==-1).astype(int)

detected = set(df.index[df["이상여부"]==1])
print("AI 가 탐지한 이상 배치:", sorted(detected))
print("실제 주입한 이상 배치 :", idx)
print("→ SPC 가 놓친 상관관계 위반도 탐지되는지 확인")

In [ ]:
# 중량-경도 산점도에 이상 배치 표시 (정상 상관선에서 벗어난 점들)
plt.figure(figsize=(6,5))
normal = df[df["이상여부"]==0]; anom = df[df["이상여부"]==1]
plt.scatter(normal["중량"], normal["경도"], alpha=0.5, label="정상")
plt.scatter(anom["중량"], anom["경도"], color="red", s=80, label="이상탐지")
plt.xlabel("정제 중량(mg)"); plt.ylabel("경도(N)")
plt.title("다변량 이상탐지: 상관관계 위반도 포착"); plt.legend(); plt.show()

## 3. Hotelling T² — 다변량 SPC의 정석
제약 현장에서 정식으로 쓰이는 **Hotelling's T²** 관리도도 만들어 봅니다.
여러 변수의 공분산을 고려한 '통합 이탈 지표'입니다.


In [ ]:
from scipy.stats import f as f_dist
Xc = df[["중량","경도","두께"]].values
mean = Xc.mean(0); cov_inv = np.linalg.inv(np.cov(Xc.T))
T2 = np.array([ (x-mean) @ cov_inv @ (x-mean) for x in Xc ])

p, m = 3, len(Xc)
ucl = (p*(m-1)/(m-p)) * f_dist.ppf(0.99, p, m-p)   # 99% 관리한계
plt.figure(figsize=(9,3))
plt.plot(T2, marker="o", ms=3)
plt.axhline(ucl, color="red", ls="--", label="T² 관리한계(99%)")
plt.scatter(np.where(T2>ucl)[0], T2[T2>ucl], color="red", s=60, zorder=5)
plt.title("Hotelling T² 다변량 관리도"); plt.legend(); plt.tight_layout(); plt.show()
print("T²가 잡은 이상 배치:", list(np.where(T2>ucl)[0]))

## 정리 & 현장 응용
- **SPC(단변량)** 는 필수지만 변수 간 관계 위반을 놓침
- **Isolation Forest / Hotelling T²** 로 다변량 이상을 포착
- 실제 적용: 타정·충전·발효·정제 공정의 실시간 품질 모니터링(PAT)
- LLM 활용 팁: "이 이상 배치의 원인을 어떤 변수가 기여했는지 분석해줘" 로 원인 진단 확장
